# Proyecto III · Minería de Datos

**ISW-911 · Caso de consultoría para Grupo Innova S.A.**

**Equipo:** Miguel, Brian, Jeremy y Cristopher.

## Introducción

Grupo Innova S.A. busca usar el historial de ventas para comprender su desempeño comercial y anticipar cambios que requieren una decisión. En este proyecto se analizarán los registros de ventas musicales de Chinook, preparados en el workshop 4 y disponibles en [data_process.csv](../data/data_process.csv). Cada fila representa una línea de factura e incluye datos como la fecha de compra, el cliente, la canción, el género, el país del cliente, la cantidad y el importe.

El punto de partida son los ocho indicadores mensuales aprobados en el Proyecto 2. Cubren el crecimiento de los ingresos y las unidades vendidas, el ingreso promedio por comprador, los clientes activos, la retención, la cobertura por países y la concentración de ingresos por género y por artista. Las fórmulas y las fuentes del Data Warehouse están en el [README](../README.md#kpis-aprobados). Para calcular la concentración en los cinco artistas principales se requiere incorporar la relación entre canción y artista desde el catálogo de Chinook.

## Objetivo del proyecto

Se desarrolla la propuesta vinculada al KPI-04 de retención. Al cierre de cada mes, se busca estimar qué clientes que compraron ese mes volverán a comprar el siguiente. Cada ejemplo representa un cliente comprador y un mes de corte. La Fase I construye la variable objetivo y sus predictores con el historial disponible. La ratificación de la pregunta por el equipo y la cobertura final de diciembre siguen pendientes.

El trabajo incluirá el análisis exploratorio y la preparación de los datos, el ajuste de al menos dos modelos supervisados y su comparación sobre un conjunto de prueba independiente. A partir del desempeño y de las variables más importantes del modelo elegido, se formularán al menos tres recomendaciones para la junta directiva.


## Fase I. Pregunta de negocio y preparación de los datos

**Responsable de esta fase, Brian.** Miguel continúa con la partición temporal,
el preprocesamiento y el entrenamiento de los dos modelos. Jeremy evalúa y
Cristopher prepara la presentación ejecutiva.

### 1. Pregunta y variable objetivo

Se desarrolla la propuesta del KPI-04 del Proyecto II. Al cierre de un mes,
¿qué clientes que compraron ese mes volverán a comprar el mes calendario siguiente?
La dirección comercial podría usar las probabilidades para priorizar un piloto
de seguimiento de compradores recientes. El modelo se evaluará con fechas futuras
antes de recomendar su uso. Un experimento con grupos comparables permitiría
medir después si el contacto genera compras adicionales.

La unidad de análisis es **cliente comprador y mes de corte**. La etiqueta
`RecompraMesSiguiente` vale 1 cuando ese cliente tiene al menos una compra en el
mes siguiente y 0 cuando se observó ese mes completo y el cliente no compró.
Es una clasificación binaria. El historial disponible termina al cierre del mes
de corte. La actividad posterior se utiliza exclusivamente para la etiqueta.

El KPI-04 de febrero, por ejemplo, divide los compradores de enero que volvieron
en febrero entre todos los compradores de enero. Ese grupo aporta las filas de
enero al modelo. Este alcance comprende a compradores del mes de corte.
La captación de clientes nuevos y la reactivación de clientes inactivos requerirían
definir otras poblaciones.

La definición se contrastó con `powerbi/documentacion/catalogo_kpis.json` del
Proyecto II. El notebook se reconstruye desde el CSV del workshop 4.
[Documentación del tablero y KPIs](https://github.com/Brianrp19/chinook_datawarehouse/blob/main/powerbi/README.md)

### 2. Carga y configuración reproducible

Ejecutar las celdas en orden desde la raíz del repositorio o desde `notebooks/`.
En Colab, colocar el CSV en `data/data_process.csv` dentro del directorio de
trabajo. Se necesitan pandas, NumPy y Matplotlib, disponibles en Colab.

**Decisión provisional sobre cobertura.** La última venta registrada es del
22 de diciembre de 2013. El CSV no certifica el cierre del extracto. Se toma
noviembre de 2013 como último mes completo para construir etiquetas y se
conservan las ventas originales. El equipo debe confirmar la cobertura con
la fuente del workshop. Si confirma diciembre completo, puede cambiar el
parámetro a `2013-12` y volver a ejecutar. Una etiqueta sin mes futuro observable
queda fuera del conjunto supervisado.


In [1]:
from pathlib import Path
import hashlib
import platform
import pandas as pd
import numpy as np

pd.set_option("display.max_rows", 80)
pd.set_option("display.max_columns", 15)

SEMILLA = 42
ULTIMO_MES_RESULTADO_COMPLETO = pd.Period("2013-11", freq="M")
ANIO_INICIO_PRUEBA_PROPUESTO = 2013
ANIO_FIN_ENTRENAMIENTO_INICIAL = 2009
VENTANA_DIAS = 90

raices_posibles = [Path.cwd(), Path.cwd().parent, Path.cwd() / "Proyecto3-mineria-datos"]
raiz = next((p for p in raices_posibles if (p / "data/data_process.csv").is_file()), None)
if raiz is None:
    raise FileNotFoundError("Colocar data/data_process.csv en la raíz del proyecto y ejecutar desde allí o desde notebooks/.")

ruta_csv = raiz / "data/data_process.csv"
ruta_tablas = raiz / "resultados/tablas"
ruta_figuras = raiz / "resultados/figuras"
ruta_tablas.mkdir(parents=True, exist_ok=True)
ruta_figuras.mkdir(parents=True, exist_ok=True)
huella_entrada = hashlib.sha256(ruta_csv.read_bytes()).hexdigest()
ventas_originales = pd.read_csv(ruta_csv)
print(f"CSV recibido: {ventas_originales.shape[0]} filas y {ventas_originales.shape[1]} columnas")
print(f"Python {platform.python_version()}, pandas {pd.__version__}, NumPy {np.__version__}")
print(f"SHA-256 de entrada: {huella_entrada}")
print(f"Último mes de resultado utilizado: {ULTIMO_MES_RESULTADO_COMPLETO}")
display(ventas_originales.head(3))


   IdLineaFactura  IdFactura  IdCancion  PrecioUnitario  Cantidad  IdCliente  \
0               1          1          2            0.99         1          2   
1               2          1          4            0.99         1          2   
2               3          2          6            0.99         1          4   

  FechaFactura  ... PaisFacturacion_Poland  PaisFacturacion_Portugal  \
0   2009-01-01  ...                      0                         0   
1   2009-01-01  ...                      0                         0   
2   2009-01-02  ...                      0                         0   

   PaisFacturacion_Spain PaisFacturacion_Sweden  PaisFacturacion_USA  \
0                      0                      0                    0   
1                      0                      0                    0   
2                      0                      0                    0   

   PaisFacturacion_United Kingdom  CategoriaPrecio  
0                               0             0.

CSV recibido: 2240 filas y 66 columnas
Python 3.12.14, pandas 3.0.1, NumPy 2.3.5
SHA-256 de entrada: ca4a465b05c5df5a80905b2592c68c911ffc98bf2a3e6126681390401331843b
Último mes de resultado utilizado: 2013-11


### 3. Estructura, tipos, cardinalidad y calidad

Cada fila del CSV representa una línea de factura. Varias líneas pueden
pertenecer a una factura y un cliente puede comprar en diferentes meses.
Se comprueban las claves y la relación importe = precio × cantidad antes de
agregar. La tabla de calidad incluye todos los campos recibidos y cuenta
también cadenas vacías o formadas por espacios.


In [2]:
perfil_columnas = pd.DataFrame({
    "TipoRecibido": ventas_originales.dtypes.astype(str),
    "ValoresDistintos": ventas_originales.nunique(dropna=True),
    "Nulos": ventas_originales.isna().sum(),
    "VaciosTexto": ventas_originales.apply(lambda s: s.astype("string").str.strip().eq("").sum()),
}).rename_axis("Columna").reset_index()
display(perfil_columnas)

ventas = ventas_originales.copy()
ventas["FechaFactura"] = pd.to_datetime(ventas["FechaFactura"], errors="raise")
columnas_requeridas = ["IdLineaFactura", "IdFactura", "IdCliente", "IdCancion", "IdGenero",
                       "FechaFactura", "PrecioUnitario", "Cantidad", "ImporteLinea", "PaisCliente"]
assert ventas[columnas_requeridas].notna().all().all(), "Revisar faltantes en los campos de construcción."
assert not ventas["PaisCliente"].str.strip().eq("").any()
assert ventas["IdLineaFactura"].is_unique, "Hay claves de línea repetidas."
assert not ventas_originales.duplicated().any(), "Revisar filas idénticas antes de continuar."
assert ventas.groupby("IdFactura")[["IdCliente", "FechaFactura"]].nunique().eq(1).all().all()
assert np.isfinite(ventas[["PrecioUnitario", "Cantidad", "ImporteLinea"]].to_numpy()).all()
assert ventas[["PrecioUnitario", "Cantidad", "ImporteLinea"]].gt(0).all().all()
assert np.allclose(ventas["ImporteLinea"], ventas["PrecioUnitario"] * ventas["Cantidad"])
assert ventas["AnioFactura"].eq(ventas["FechaFactura"].dt.year).all()
assert ventas["MesFactura"].eq(ventas["FechaFactura"].dt.month).all()
assert np.allclose(ventas["DuracionMinutos"], ventas["DuracionMilisegundos"] / 60000)

ventas["MesVenta"] = ventas["FechaFactura"].dt.to_period("M")
meses = pd.period_range(ventas["MesVenta"].min(), ventas["MesVenta"].max(), freq="M")
assert set(ventas["MesVenta"]) == set(meses), "Hay meses sin registros cuya cobertura se debe confirmar."
assert meses.min() < ULTIMO_MES_RESULTADO_COMPLETO <= meses.max()

resumen_entrada = pd.Series({
    "LineasFactura": len(ventas), "ColumnasRecibidas": ventas_originales.shape[1],
    "Facturas": ventas["IdFactura"].nunique(), "Clientes": ventas["IdCliente"].nunique(),
    "ContenidosVendidos": ventas["IdCancion"].nunique(),
    "NulosRecibidos": int(perfil_columnas["Nulos"].sum()),
    "VaciosTexto": int(perfil_columnas["VaciosTexto"].sum()),
    "ImporteTotalUM": round(ventas["ImporteLinea"].sum(), 2),
    "PrimeraVenta": str(ventas["FechaFactura"].min().date()),
    "UltimaVenta": str(ventas["FechaFactura"].max().date()),
    "MesesConVentas": len(meses),
}, name="Valor")
display(resumen_entrada.to_frame())


                           Columna TipoRecibido  ValoresDistintos  Nulos  \
0                   IdLineaFactura        int64              2240      0   
1                        IdFactura        int64               412      0   
2                        IdCancion        int64              1984      0   
3                   PrecioUnitario      float64                 2      0   
4                         Cantidad        int64                 1      0   
5                        IdCliente        int64                59      0   
6                     FechaFactura          str               354      0   
7                          Cancion          str              1888      0   
8                         IdGenero        int64                24      0   
9             DuracionMilisegundos        int64              1846      0   
10                     PaisCliente          str                24      0   
11                    ImporteLinea      float64                 2      0   
12          

                         Valor
LineasFactura             2240
ColumnasRecibidas           66
Facturas                   412
Clientes                    59
ContenidosVendidos        1984
NulosRecibidos               0
VaciosTexto                  0
ImporteTotalUM          2328.6
PrimeraVenta        2009-01-01
UltimaVenta         2013-12-22
MesesConVentas              60

**Tratamiento aplicado.** El archivo recibido tiene 2.240 líneas, 412 facturas,
59 clientes, 66 columnas y 2.328,60 UM de ingresos. Los campos recibidos están
completos y las claves de línea son únicas. Se conserva cada línea, se convierte
la fecha a tipo temporal y se verifican año, mes y duración. La falta de una compra
posterior se convierte en 0 únicamente dentro de meses de resultado observables.

Los indicadores de género y país y las columnas normalizadas provienen del
workshop. Para la tabla nueva se usan los campos originales y agregaciones
temporales. Miguel ajustará las transformaciones que aprendan parámetros dentro
de cada bloque de entrenamiento, conforme a la guía de prevención de fuga de
información. [Preparación y fuga de información en scikit-learn](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage)

### 4. De líneas de factura a compradores por mes

Se suman los importes y las cantidades de las líneas. El número de facturas y
compradores se cuenta por identificador distinto para conservar su significado.
La siguiente tabla deja trazables todos los meses e indica el límite de cobertura
adoptado. En las gráficas de desarrollo se usan meses anteriores al año de prueba.


In [3]:
compras_cliente_mes = ventas.groupby(["MesVenta", "IdCliente"], as_index=False).agg(
    FacturasMes=("IdFactura", "nunique"),
    GastoMesUM=("ImporteLinea", "sum"),
    UnidadesMes=("Cantidad", "sum"),
    GenerosMes=("IdGenero", "nunique"),
)
resumen_mensual = ventas.groupby("MesVenta").agg(
    Compradores=("IdCliente", "nunique"), Facturas=("IdFactura", "nunique"),
    Unidades=("Cantidad", "sum"), IngresosUM=("ImporteLinea", "sum"),
    PrimeraVenta=("FechaFactura", "min"), UltimaVenta=("FechaFactura", "max"),
).reindex(meses)
resumen_mensual.index.name = "Mes"
resumen_mensual["CoberturaResultadoAdoptada"] = resumen_mensual.index <= ULTIMO_MES_RESULTADO_COMPLETO
assert np.isclose(resumen_mensual["IngresosUM"].sum(), ventas["ImporteLinea"].sum())
assert resumen_mensual["Facturas"].sum() == ventas["IdFactura"].nunique()
display(resumen_mensual.head(3))
display(resumen_mensual.tail(3))


         Compradores  Facturas  Unidades  IngresosUM PrimeraVenta UltimaVenta  \
Mes                                                                             
2009-01            6         6        36       35.64   2009-01-01  2009-01-19   
2009-02            7         7        38       37.62   2009-02-01  2009-02-19   
2009-03            7         7        38       37.62   2009-03-04  2009-03-22   

         CoberturaResultadoAdoptada  
Mes                                  
2009-01                        True  
2009-02                        True  
2009-03                        True  

         Compradores  Facturas  Unidades  IngresosUM PrimeraVenta UltimaVenta  \
Mes                                                                             
2013-10            7         7        38       37.62   2013-10-03  2013-10-21   
2013-11            7         7        38       49.62   2013-11-03  2013-11-21   
2013-12            7         7        38       38.62   2013-12-04  2013-12-22   

         CoberturaResultadoAdoptada  
Mes                                  
2013-10                        True  
2013-11                        True  
2013-12                       False  

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
mensual_desarrollo = resumen_mensual.loc[resumen_mensual.index.year < ANIO_INICIO_PRUEBA_PROPUESTO]
fig, ejes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
fechas = mensual_desarrollo.index.to_timestamp()
ejes[0].plot(fechas, mensual_desarrollo["Compradores"], color="#24557A")
ejes[0].set_ylabel("Compradores distintos")
ejes[0].set_title("Compradores por mes antes del periodo de prueba")
ejes[1].plot(fechas, mensual_desarrollo["IngresosUM"], color="#168B87")
ejes[1].set_ylabel("Ingresos (UM)")
ejes[1].set_xlabel("Mes de venta")
fig.tight_layout()
fig.savefig(ruta_figuras / "eda_compras_mensuales.png", dpi=160, bbox_inches="tight")
plt.show()


### 5. Construcción de la etiqueta de recompra

Para cada cliente que compró en el mes de corte, se busca su presencia en el
mes calendario siguiente. Se usa una unión de claves únicas cliente-mes.
El último mes de resultado admitido se controla en la configuración.
Diciembre de 2013 como mes de corte requeriría observar enero de 2014.

El cociente de positivos entre compradores de una cohorte reconstruye el KPI-04
del mes de resultado. Los conteos acompañan las tasas porque cada cohorte es
pequeña. La frecuencia global de la etiqueta pondera por compradores de cada
cohorte, mientras que el tablero presenta un porcentaje para cada mes.


In [4]:
cohortes = compras_cliente_mes[["MesVenta", "IdCliente"]].rename(columns={"MesVenta": "MesCorte"})
cohortes["MesResultado"] = cohortes["MesCorte"] + 1
cohortes = cohortes.loc[cohortes["MesResultado"] <= ULTIMO_MES_RESULTADO_COMPLETO].copy()

presencia_siguiente = compras_cliente_mes[["MesVenta", "IdCliente"]].rename(columns={"MesVenta": "MesResultado"})
presencia_siguiente["RecompraMesSiguiente"] = 1
cohortes = cohortes.merge(presencia_siguiente, on=["MesResultado", "IdCliente"],
                          how="left", validate="one_to_one")
cohortes["RecompraMesSiguiente"] = cohortes["RecompraMesSiguiente"].fillna(0).astype("int8")
cohortes = cohortes.sort_values(["MesCorte", "IdCliente"]).reset_index(drop=True)

kpi_retencion = cohortes.groupby("MesResultado").agg(
    CompradoresMesAnterior=("IdCliente", "size"),
    ClientesRetenidos=("RecompraMesSiguiente", "sum"),
)
kpi_retencion["KPI04_RetencionPct"] = 100 * kpi_retencion["ClientesRetenidos"] / kpi_retencion["CompradoresMesAnterior"]
assert cohortes["RecompraMesSiguiente"].isin([0, 1]).all()
assert not cohortes.duplicated(["IdCliente", "MesCorte"]).any()
assert len(cohortes) == kpi_retencion["CompradoresMesAnterior"].sum()

cohortes_desarrollo = cohortes.loc[cohortes["MesResultado"].dt.year < ANIO_INICIO_PRUEBA_PROPUESTO]
distribucion_desarrollo = cohortes_desarrollo["RecompraMesSiguiente"].value_counts().reindex([0, 1], fill_value=0).to_frame("Casos")
distribucion_desarrollo["Porcentaje"] = 100 * distribucion_desarrollo["Casos"] / len(cohortes_desarrollo)
display(distribucion_desarrollo)
print(f"Casos con desenlace observable: {len(cohortes)}")
print(f"Casos anteriores a {ANIO_INICIO_PRUEBA_PROPUESTO}: {len(cohortes_desarrollo)}")
print(f"Casos reservables desde {ANIO_INICIO_PRUEBA_PROPUESTO}: {len(cohortes) - len(cohortes_desarrollo)}")
display(kpi_retencion.loc[kpi_retencion.index.year < ANIO_INICIO_PRUEBA_PROPUESTO].head())


                      Casos  Porcentaje
RecompraMesSiguiente                   
0                       292   89.846154
1                        33   10.153846

              CompradoresMesAnterior  ClientesRetenidos  KPI04_RetencionPct
MesResultado                                                               
2009-02                            6                  1           16.666667
2009-03                            7                  1           14.285714
2009-04                            7                  1           14.285714
2009-05                            7                  1           14.285714
2009-06                            7                  1           14.285714

Casos con desenlace observable: 398
Casos anteriores a 2013: 325
Casos reservables desde 2013: 73


In [ ]:
retencion_desarrollo = kpi_retencion.loc[kpi_retencion.index.year < ANIO_INICIO_PRUEBA_PROPUESTO]
fig, ejes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
fechas = retencion_desarrollo.index.to_timestamp()
ejes[0].plot(fechas, retencion_desarrollo["CompradoresMesAnterior"], label="Compradores del mes anterior", color="#24557A")
ejes[0].plot(fechas, retencion_desarrollo["ClientesRetenidos"], label="Compraron otra vez", color="#168B87")
ejes[0].set_ylabel("Clientes")
ejes[0].set_title("Recompra mensual en desarrollo")
ejes[0].legend()
ejes[1].plot(fechas, retencion_desarrollo["KPI04_RetencionPct"], color="#B34B37")
ejes[1].set_ylabel("KPI-04 (%)")
ejes[1].set_xlabel("Mes de resultado")
fig.tight_layout()
fig.savefig(ruta_figuras / "eda_retencion_mensual.png", dpi=160, bbox_inches="tight")
plt.show()


### 6. Ejemplo rastreable a las ventas originales

En enero de 2009 compraron seis clientes. Solo el cliente 2 vuelve en febrero.
La cohorte de enero aporta seis ejemplos, uno positivo y cinco negativos.
El KPI-04 de febrero es 1/6 = 16,67 %. La tabla siguiente muestra ese cruce y
las líneas originales del cliente 2 que permiten comprobarlo.


In [5]:
ejemplo_enero = cohortes.loc[cohortes["MesCorte"] == pd.Period("2009-01", "M")].copy()
display(ejemplo_enero)
display(ventas.loc[
    ventas["IdCliente"].eq(2) & ventas["MesVenta"].isin(pd.period_range("2009-01", "2009-02", freq="M")),
    ["IdLineaFactura", "IdFactura", "IdCliente", "FechaFactura", "Cantidad", "ImporteLinea"],
])
assert len(ejemplo_enero) == 6
assert ejemplo_enero["RecompraMesSiguiente"].sum() == 1
assert ejemplo_enero.loc[ejemplo_enero["RecompraMesSiguiente"].eq(1), "IdCliente"].tolist() == [2]
assert np.isclose(kpi_retencion.loc[pd.Period("2009-02", "M"), "KPI04_RetencionPct"], 100 / 6)


  MesCorte  IdCliente MesResultado  RecompraMesSiguiente
0  2009-01          2      2009-02                     1
1  2009-01          4      2009-02                     0
2  2009-01          8      2009-02                     0
3  2009-01         14      2009-02                     0
4  2009-01         23      2009-02                     0
5  2009-01         37      2009-02                     0

    IdLineaFactura  IdFactura  IdCliente FechaFactura  Cantidad  ImporteLinea
0                1          1          2   2009-01-01         1          0.99
1                2          1          2   2009-01-01         1          0.99
59              60         12          2   2009-02-11         1          0.99
60              61         12          2   2009-02-11         1          0.99
61              62         12          2   2009-02-11         1          0.99
62              63         12          2   2009-02-11         1          0.99
63              64         12          2   2009-02-11         1          0.99
64              65         12          2   2009-02-11         1          0.99
65              66         12          2   2009-02-11         1          0.99
66              67         12          2   2009-02-11         1          0.99
67              68         12          2   2009-02-11         1          0.99
68              69         12          2   2009-02-11         1 

### 7. Variables conocidas al cierre del mes

Se calculan medidas de recencia, frecuencia, gasto y diversidad con una ventana
de **90 días calendario**, cuyo extremo derecho es el primer día del siguiente
mes, excluido. La recencia usa días completos desde la última compra hasta la
fecha de corte. La antigüedad se mide desde la primera compra que aparece en el
extracto. El país corresponde a la última venta observada hasta ese cierre.

El comienzo del archivo limita la historia disponible. `DiasVentanaObservados`
registra cuántos de los 90 días están cubiertos por el extracto.
`AntiguedadObservadaDias` permite distinguir clientes recién observados.
La cobertura de los meses anteriores al último se asume continua a partir de
las exportaciones del workshop. La ausencia de ventas por sí sola no certifica
la cobertura de una fuente.

La función recibe solo ventas y las claves de los casos. La etiqueta se adjunta
después del cálculo de los predictores. Los precios normalizados, los indicadores
del workshop y `CategoriaPrecio` se mantienen en el CSV fuente para trazabilidad.
La lista explícita de predictores para Miguel aparece al final de esta fase.


In [6]:
def construir_predictores(ventas_fuente, claves):
    inicio_extracto = ventas_fuente["FechaFactura"].min().normalize()
    filas = []
    for mes, grupo in claves.groupby("MesCorte", sort=True):
        fin_exclusivo = (mes + 1).start_time
        fecha_corte = fin_exclusivo - pd.Timedelta(days=1)
        inicio_ventana = fin_exclusivo - pd.Timedelta(days=VENTANA_DIAS)
        historia = ventas_fuente.loc[ventas_fuente["FechaFactura"] < fin_exclusivo]
        reciente = historia.loc[historia["FechaFactura"] >= inicio_ventana]
        medidas = reciente.groupby("IdCliente").agg(
            Facturas90d=("IdFactura", "nunique"),
            Gasto90dUM=("ImporteLinea", "sum"),
            Unidades90d=("Cantidad", "sum"),
            Generos90d=("IdGenero", "nunique"),
        )
        ultima = historia.sort_values(["FechaFactura", "IdLineaFactura"]).groupby("IdCliente").tail(1).set_index("IdCliente")
        primera = historia.groupby("IdCliente")["FechaFactura"].min()
        bloque = grupo[["IdCliente", "MesCorte", "MesResultado"]].set_index("IdCliente").join(medidas)
        bloque["FechaCorte"] = fecha_corte
        bloque["RecenciaDias"] = (fecha_corte - ultima["FechaFactura"].dt.normalize()).dt.days
        bloque["AntiguedadObservadaDias"] = (fecha_corte - primera.dt.normalize()).dt.days
        bloque["DiasVentanaObservados"] = (fin_exclusivo - max(inicio_ventana, inicio_extracto)).days
        bloque["PaisCliente"] = ultima["PaisCliente"]
        filas.append(bloque.reset_index())
    return pd.concat(filas, ignore_index=True).sort_values(["MesCorte", "IdCliente"]).reset_index(drop=True)

predictores_candidatos = construir_predictores(ventas, cohortes[["IdCliente", "MesCorte", "MesResultado"]])
datos_cliente_mes = predictores_candidatos.merge(
    cohortes[["IdCliente", "MesCorte", "RecompraMesSiguiente"]],
    on=["IdCliente", "MesCorte"], how="left", validate="one_to_one",
)
assert len(datos_cliente_mes) == len(cohortes)
assert datos_cliente_mes.notna().all().all()
assert datos_cliente_mes["RecenciaDias"].between(0, 30).all()
assert datos_cliente_mes["DiasVentanaObservados"].between(1, VENTANA_DIAS).all()
display(datos_cliente_mes.head(6))


   IdCliente MesCorte MesResultado  Facturas90d  Gasto90dUM  Unidades90d  \
0          2  2009-01      2009-02            1        1.98            2   
1          4  2009-01      2009-02            1        3.96            4   
2          8  2009-01      2009-02            1        5.94            6   
3         14  2009-01      2009-02            1        8.91            9   
4         23  2009-01      2009-02            1       13.86           14   
5         37  2009-01      2009-02            1        0.99            1   

   Generos90d FechaCorte  RecenciaDias  AntiguedadObservadaDias  \
0           1 2009-01-31            30                       30   
1           1 2009-01-31            29                       29   
2           1 2009-01-31            28                       28   
3           3 2009-01-31            25                       25   
4           6 2009-01-31            20                       20   
5           1 2009-01-31            12                       12  